# Structured Summarization

**Structured summarization** refers to generating summaries in a predefined, organized format—rather than free-flowing prose—so the output follows a consistent schema that's predictable and easy to parse or reuse.

## How it differs from regular summarization

| Regular summarization | Structured summarization |
|---|---|
| Free-form paragraph(s) | Fixed schema (JSON, tables, bullet lists, sections) |
| Varies in length/format each time | Consistent structure every time |
| Meant for human reading | Meant for both humans and downstream systems (APIs, databases, UI) |

## Common formats it can take

- **JSON/dict output** — e.g., `{"title": ..., "key_points": [...], "sentiment": ..., "action_items": [...]}`
- **Fixed sections** — e.g., always producing "Background," "Findings," "Conclusion" for a research paper
- **Tables** — extracting entities like dates, names, and amounts into rows/columns
- **Bullet-point templates** — e.g., meeting notes always broken into "Decisions," "Action Items," "Open Questions"
- **Key-value extraction** — pulling specific fields out of a document (invoice number, due date, total)

## Why it's used

1. **Downstream automation** — structured output can be fed directly into another program, database, or UI component without additional parsing logic.
2. **Consistency** — useful when summarizing many documents (e.g., hundreds of customer reviews) and you need every output to have the same shape for comparison or aggregation.
3. **Reduced hallucination risk in some fields** — constraining the model to specific slots (like "date" or "amount") can reduce rambling or irrelevant content.
4. **Easier evaluation** — structured outputs are simpler to score against ground truth (e.g., "did it correctly extract the invoice total?") than free-text summaries.

## How it's typically achieved

- **Prompt engineering** — explicitly instructing the model to output in a given format (e.g., "Summarize this in JSON with fields: summary, sentiment, keywords").
- **Function calling / tool schemas** — many LLM APIs (including Claude's) let you define a JSON schema and the model fills it in, guaranteeing valid structure.
- **Constrained decoding / grammars** — some systems force the model's token generation to only produce outputs matching a grammar (e.g., valid JSON).
- **Few-shot examples** — showing the model examples of the input-to-structured-output mapping.

## Example use cases

- Summarizing legal contracts into structured fields (parties, obligations, dates, penalties)
- Turning customer support transcripts into `{issue, resolution, sentiment, follow_up_needed}`
- Extracting structured metadata from scientific papers (methods, results, limitations)
- Meeting transcript → agenda items, decisions, and owners

If you're working on a specific use case (like building an app that needs this), I'm happy to help design a schema or prompt for it.

## Build document summarizer with JSON output and Extract key entities and sentiment as well using openai.

### Example 1: Using Class

In [2]:
from openai import OpenAI
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv
import os

# Load API Key
load_dotenv()

client = OpenAI()


# ----------------------------
# Output Schema
# ----------------------------

class Entities(BaseModel):
    people: List[str]
    organizations: List[str]
    locations: List[str]
    dates: List[str]
    products: List[str]


class Sentiment(BaseModel):
    label: str          # Positive, Neutral, Negative
    confidence: float   # 0-1


class DocumentSummary(BaseModel):
    title: str
    summary: str
    key_points: List[str]
    entities: Entities
    sentiment: Sentiment


# ----------------------------
# Function
# ----------------------------

def summarize_document(document: str):

    response = client.responses.parse(
        model="gpt-5",
        input=[
            {
                "role": "system",
                "content": (
                    "You are an expert document analyst.\n"
                    "Summarize documents accurately.\n"
                    "Extract important entities.\n"
                    "Determine overall sentiment.\n"
                    "Return only structured JSON."
                )
            },
            {
                "role": "user",
                "content": document
            }
        ],
        text_format=DocumentSummary
    )

    return response.output_parsed


# ----------------------------
# Example Document
# ----------------------------

document = """
OpenAI announced a partnership with Microsoft to improve enterprise AI solutions.

The collaboration will expand Azure AI services globally beginning in September 2026.

The announcement received positive feedback from customers and investors because
of improved scalability and security.
"""


result = summarize_document(document)

print(result.model_dump_json(indent=4))

{
    "title": "OpenAI–Microsoft partnership to expand Azure AI globally in September 2026",
    "summary": "OpenAI and Microsoft announced a partnership to enhance enterprise AI solutions and expand Azure AI services worldwide starting in September 2026. The news was positively received by customers and investors, citing improved scalability and security.",
    "key_points": [
        "OpenAI and Microsoft formed a partnership focused on enterprise AI.",
        "Azure AI services will be expanded globally.",
        "Rollout begins in September 2026.",
        "Positive reception from customers and investors due to scalability and security improvements."
    ],
    "entities": {
        "people": [],
        "organizations": [
            "OpenAI",
            "Microsoft"
        ],
        "locations": [],
        "dates": [
            "September 2026"
        ],
        "products": [
            "Azure AI services"
        ]
    },
    "sentiment": {
        "label": "positive",
 

### Example 2: Using JSON Schema object


In [3]:
"""
Document Summarizer with Structured JSON Output (OpenAI API)
--------------------------------------------------------------
Summarizes a document and extracts:
  - summary (concise text)
  - key_points (bullet list)
  - entities (people, organizations, locations, dates, misc)
  - sentiment (overall label + score + rationale)

Uses OpenAI's Structured Outputs feature (response_format with a JSON
schema) so the model is constrained to return valid, well-typed JSON
every time -- no manual parsing/regex needed.

Requirements:
    pip install openai

Set your API key as an environment variable before running:
    export OPENAI_API_KEY="sk-..."
"""

import os
import json
from openai import OpenAI

# ---------------------------------------------------------------------
# 1. JSON schema definition
# ---------------------------------------------------------------------
# This schema is passed to the model via `response_format`, guaranteeing
# the output matches this exact structure (OpenAI validates it server-side).

SUMMARY_SCHEMA = {
    "name": "document_analysis",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "summary": {
                "type": "string",
                "description": "A concise 3-5 sentence summary of the document."
            },
            "key_points": {
                "type": "array",
                "items": {"type": "string"},
                "description": "The most important points/findings, as short bullet phrases."
            },
            "entities": {
                "type": "object",
                "properties": {
                    "people": {"type": "array", "items": {"type": "string"}},
                    "organizations": {"type": "array", "items": {"type": "string"}},
                    "locations": {"type": "array", "items": {"type": "string"}},
                    "dates": {"type": "array", "items": {"type": "string"}},
                    "misc": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Other notable named entities (products, laws, events, etc.)"
                    }
                },
                "required": ["people", "organizations", "locations", "dates", "misc"],
                "additionalProperties": False
            },
            "sentiment": {
                "type": "object",
                "properties": {
                    "label": {
                        "type": "string",
                        "enum": ["positive", "negative", "neutral", "mixed"]
                    },
                    "score": {
                        "type": "number",
                        "description": "Sentiment polarity from -1.0 (very negative) to 1.0 (very positive)."
                    },
                    "rationale": {
                        "type": "string",
                        "description": "One-sentence explanation for the sentiment label."
                    }
                },
                "required": ["label", "score", "rationale"],
                "additionalProperties": False
            }
        },
        "required": ["summary", "key_points", "entities", "sentiment"],
        "additionalProperties": False
    }
}

SYSTEM_PROMPT = (
    "You are a precise document analysis assistant. Read the provided "
    "document and extract a summary, key points, named entities, and the "
    "overall sentiment. Only include entities that are explicitly present "
    "in the text. Be objective and factual in the summary; do not add "
    "information that isn't in the source document."
)


class DocumentSummarizer:
    def __init__(self, api_key: str | None = None, model: str = "gpt-4o-mini"):
        """
        api_key: OpenAI API key. If None, reads from OPENAI_API_KEY env var.
        model:   Any OpenAI model that supports structured outputs
                  (e.g. gpt-4o, gpt-4o-mini, gpt-4.1, gpt-4.1-mini).
        """
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model = model

    def summarize(self, document_text: str) -> dict:
        """
        Sends the document to the model and returns a parsed dict matching
        SUMMARY_SCHEMA: {summary, key_points, entities, sentiment}
        """
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Document:\n\n{document_text}"}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": SUMMARY_SCHEMA
            },
            temperature=0.2
        )

        raw_json = response.choices[0].message.content
        return json.loads(raw_json)

    def summarize_file(self, file_path: str) -> dict:
        """Convenience method: reads a plain-text file and summarizes it."""
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        return self.summarize(text)


# ---------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------
if __name__ == "__main__":
    sample_document = """
    Apple Inc. announced record quarterly revenue of $95 billion on
    October 28, 2025, driven largely by strong iPhone 17 sales in
    the United States and China. CEO Tim Cook praised the team's
    execution, calling it "one of the best quarters in company history."
    However, some analysts at Morgan Stanley expressed concern that
    growth in Europe was slowing, citing weaker consumer spending in
    Germany and France. The company's stock rose 4% in after-hours
    trading following the announcement.
    """

    summarizer = DocumentSummarizer(model="gpt-4o-mini")
    result = summarizer.summarize(sample_document)

    print(json.dumps(result, indent=2))


{
  "summary": "Apple Inc. reported a record quarterly revenue of $95 billion on October 28, 2025, primarily due to robust sales of the iPhone 17 in the U.S. and China. CEO Tim Cook highlighted the quarter as one of the best in the company's history. Despite this success, analysts at Morgan Stanley raised concerns about slowing growth in Europe, particularly due to decreased consumer spending in Germany and France. Following the announcement, Apple's stock increased by 4% in after-hours trading.",
  "key_points": [
    "Apple Inc. reported $95 billion in quarterly revenue",
    "Strong iPhone 17 sales in the U.S. and China",
    "CEO Tim Cook praised the quarter's performance",
    "Concerns about slowing growth in Europe",
    "Apple's stock rose 4% in after-hours trading"
  ],
  "entities": {
    "people": [
      "Tim Cook"
    ],
    "organizations": [
      "Apple Inc.",
      "Morgan Stanley"
    ],
    "locations": [
      "United States",
      "China",
      "Germany",
      "